[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dandi/example-notebooks/blob/master/000055/BruntonLab/peterson21/Fig_coarse_labels.ipynb)

## Installing requirements

The cell below installs every Python package needed to run this notebook, at fully pinned versions, using [`uv`](https://github.com/astral-sh/uv) for fast resolution. In Colab the cell is collapsed by default — click the ▶ button to run it.

In [ ]:
#@title Installing requirements (click ▶ to run) { display-mode: "form" }
# Colab provides Python 3.13. We install with `uv --system` because Colab's
# kernel runs outside a virtualenv. All versions (direct + transitive) are
# pinned below so the notebook is reproducible regardless of resolver drift.
!pip install -q uv
!uv pip install --system \
    "acres==0.5.0" \
    "aiohappyeyeballs==2.7.1" \
    "aiohttp==3.14.3" \
    "aiosignal==1.4.0" \
    "annotated-types==0.8.0" \
    "arrow==1.4.0" \
    "attrs==26.1.0" \
    "bids-validator-deno==3.0.2" \
    "bidsschematools==1.2.7" \
    "blessed==1.50.0" \
    "certifi==2026.7.22" \
    "cffi==2.1.1" \
    "charset-normalizer==3.4.9" \
    "ci-info==0.4.0" \
    "click==8.5.0" \
    "click-didyoumean==0.3.1" \
    "cloudpickle==3.1.2" \
    "contourpy==1.3.3" \
    "cryptography==50.0.1" \
    "cycler==0.12.1" \
    "dandi==0.81.0" \
    "dandischema==0.14.0" \
    "deno==2.9.7" \
    "dnspython==2.9.0" \
    "donfig==0.8.1.post1" \
    "email-validator==2.3.0" \
    "etelemetry==0.3.1" \
    "fasteners==0.20" \
    "fonttools==4.64.0" \
    "fqdn==1.5.1" \
    "frozenlist==1.8.0" \
    "fscacher==0.5.0" \
    "fsspec==2025.12.0" \
    "google-crc32c==1.8.0" \
    "h5py==3.16.0" \
    "hdmf==6.2.0" \
    "humanize==4.16.0" \
    "idna==3.19" \
    "interleave==0.3.0" \
    "isodate==0.7.2" \
    "isoduration==20.11.0" \
    "jaraco-classes==3.4.0" \
    "jaraco-context==6.1.2" \
    "jaraco-functools==4.6.0" \
    "jeepney==0.9.0" \
    "jinja2==3.1.6" \
    "jinxed==2.1.0" \
    "joblib==1.6.0" \
    "jsonpointer==3.1.1" \
    "jsonschema==4.26.0" \
    "jsonschema-specifications==2025.9.1" \
    "keyring==25.7.0" \
    "keyrings-alt==5.0.2" \
    "kiwisolver==1.5.1" \
    "markupsafe==3.0.3" \
    "matplotlib==3.10.0" \
    "ml-dtypes==0.6.0" \
    "more-itertools==10.8.0" \
    "multidict==6.7.1" \
    "natsort==8.4.0" \
    "nibabel==5.4.2" \
    "nilearn==0.13.1" \
    "numcodecs==0.17.0" \
    "numpy==2.1.3" \
    "nwbinspector==0.7.2" \
    "packaging==26.3" \
    "pandas==2.2.3" \
    "pillow==11.3.0" \
    "platformdirs==4.11.7" \
    "propcache==0.5.2" \
    "pycparser==3.0" \
    "pycryptodomex==3.23.0" \
    "pydantic==2.13.5" \
    "pydantic-core==2.46.5" \
    "pydantic-settings==2.15.0" \
    "pynwb==4.2.0" \
    "pyout==0.8.1" \
    "pyparsing==3.3.2" \
    "python-dateutil==2.9.0.post0" \
    "python-dotenv==1.2.3" \
    "pytz==2025.2" \
    "pyyaml==6.0.3" \
    "referencing==0.37.0" \
    "remfile==0.1.15" \
    "requests==2.32.4" \
    "rfc3339-validator==0.1.4" \
    "rfc3987==1.3.8" \
    "rpds-py==2026.6.3" \
    "ruamel-yaml==0.19.1" \
    "scikit-learn==1.6.1" \
    "scipy==1.16.3" \
    "seaborn==0.13.2" \
    "secretstorage==3.5.0" \
    "semantic-version==2.10.0" \
    "six==1.17.0" \
    "tenacity==9.1.4" \
    "tensorstore==0.1.85" \
    "threadpoolctl==3.6.0" \
    "tqdm==4.67.3" \
    "typing-extensions==4.16.0" \
    "typing-inspection==0.4.4" \
    "tzdata==2026.3" \
    "uri-template==1.3.0" \
    "urllib3==2.5.0" \
    "wcwidth==0.8.3" \
    "webcolors==25.10.0" \
    "yarl==1.24.5" \
    "zarr==3.1.5" \
    "zarr-checksum==0.4.7"
!curl -sL -o plot_utils.py https://raw.githubusercontent.com/dandi/example-notebooks/master/000055/BruntonLab/peterson21/plot_utils.py

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already loaded in the kernel. Go to **Runtime → Restart session**, then **Run all cells below** (skip this install cell on re-run).

# Plot example coarse behavioral labelling trace

Replicates example coarse behavior labelling trace for one recording day. Note that the figure from the data paper combined the targeted (targeted=True) and untargeted (both first_val=True and first_val=False) behavior labels.

In [1]:
from dandi.dandiapi import DandiAPIClient

from plot_utils import open_nwb_stream, prune_clabels, plot_clabels

## Set parameters

In [2]:
targ_tlims = [13, 17]  # targeted window to plot (in hours)
targeted = True  # plot targeted window (True) or whole day (False)
targ_label = 'Computer/phone'

## Load NWB data file

In [ ]:
with DandiAPIClient() as client:
    asset = client.get_dandiset("000055", "draft").get_asset_by_path(
        "sub-01/sub-01_ses-4_behavior+ecephys.nwb"
    )
    s3_path = asset.get_content_url(follow_redirects=1, strip_query=True)

with open_nwb_stream(s3_path) as io:
    nwb = io.read()
    clabels_orig = nwb.intervals['epochs'].to_dataframe()

## Select coarse labels based on user parameters

In [ ]:
label_col_d = {
    'Other activity': 0,
    'Computer/phone': 1,
    'Eat': 2,
    'TV': 3,
    'Talk': 4
}

clabels, uni_labs = prune_clabels(clabels_orig, targeted,
                                  targ_tlims, None,
                                  targ_label)

## Plot labels over time

In [ ]:
fig = plot_clabels(clabels, uni_labs, targeted, None, targ_tlims, targlab_colind=label_col_d[targ_label])


# # Save figure
# fig_sp = ''
# targ_label_out = '_'.join(targ_label.split(' '))
# targ_label_out = '_'.join(targ_label_out.split('/'))
# fig.savefig(fig_sp+targ_label_out+'_targeted_trace.png',
#             format='png',  transparent= True,dpi=150,
#             bbox_inches = 'tight', pad_inches = 0.01,
#             )